# Main Results — AMI

Reproduces the AMI main-results table in Section 4.2 of the manuscript. The pipeline runs (in order):

1. **Training-free baselines** — random, weighted random, single base models, oracle (`src.training.eval_baselines`).
2. **ROVER and weighted-ROVER** — hypothesis combination via word-level voting (`src.training.rover`).
3. **Trained methods** — MLP-pool baseline and the proposed hierarchical transformer (`src.training.train` + `src.training.evaluate`).

Total runtime on L4: ~1.5 h (mostly the two trained methods, ~30–40 min each).


## 1. Initial setup (clone + uv env + secrets)

GPU recommendation:
- **L4** (default): best price/performance for these notebooks. ~1.5–3× faster than T4 with no per-second penalty.
- **A100**: ~2× faster than L4, ~2× the price. Use only if you want the full 50-epoch run in under 30 minutes.
- **T4**: works but slow; halve `batch_size` if you OOM.


In [ ]:
! git clone https://github.com/huseyin-karaca/s2t-tr-dev
%cd /content/s2t-tr-dev

from google.colab import files
files.view('/content/s2t-tr-dev')

!make create_environment
!make requirements

from google.colab import userdata
import os
os.environ['GITHUB_TOKEN'] = userdata.get('GitHubPAT')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

!gh auth status
!git config --global user.name 'huseyin-karaca'
!git config --global user.email 'huseyinkaraccca@gmail.com'

from google.colab import drive
drive.mount('/content/drive')


## 2. Bring in the AMI interim features from Drive

Reuses the per-expert interim parquets already on Drive (no re-extraction). Then runs the patched `preprocess.py` so that the combined parquet carries the per-system transcription columns needed for ROVER.

> **Adjust `DRIVE_INTERIM_AMI` below** to your actual Drive folder path.


In [ ]:
DRIVE_INTERIM_AMI = '/content/drive/MyDrive/s2t-tr-dev/data/interim/edinburghcstr_ami'

!mkdir -p data/interim/edinburghcstr_ami
!cp -v "$DRIVE_INTERIM_AMI"/*.parquet data/interim/edinburghcstr_ami/
!ls -lh data/interim/edinburghcstr_ami/


In [ ]:
!uv run python -m src.data.preprocess \
    --dataset edinburghcstr/ami \
    --output_name combined_features_with_transcripts.parquet


## 3. Smoke test (1 epoch, limited batches)


In [ ]:
!uv run python -m src.training.train \
    --parquet-path data/processed/edinburghcstr_ami/combined_features_with_transcripts.parquet \
    --arch hierarchical_transformer \
    --batch-size 8 --num-workers 2 \
    --max-epochs 1 --limit-batches 8 \
    --experiment-name smoke_test \
    --log-dir reports/main_results/ami


## 4. Run the main-results pipeline

Training-free + ROVER + 2 trained methods. The orchestrator writes the combined `main_results.json` incrementally so it is safe to interrupt and resume by re-running with `--skip-training-free --skip-rover`.

> **A100 settings**: `max_epochs 30`, `batch_size 16` in the JSON config.
> **T4 settings**: `batch_size 4`.


In [ ]:
!uv run python -m src.experiments.main_results \
    --config configs/main_results_ami.json \
    --output-dir reports/main_results/ami


## 5. Render the main-results table

Prints one row per method (training-free baselines, ROVER, trained methods). This is the data backing the main-results table in the manuscript.


In [ ]:
import json
from pathlib import Path

results = json.loads(Path('reports/main_results/ami/main_results.json').read_text())
rows = []

tf = results.get('training_free', {}).get('results', {}).get('test', {})
for name in ['random', 'weighted_random', 'single_hubert', 'single_whisper',
             'single_wav2vec2', 'oracle']:
    s = tf.get(name)
    if s and 'wer_mean' in s:
        rows.append((name, s['wer_mean'], s['wer_sem']))

rover = results.get('rover', {}).get('results', {}).get('test', {})
for name in ['rover', 'weighted_rover']:
    s = rover.get(name)
    if s:
        rows.append((name, s['wer_mean'], s['wer_sem']))

for name, entry in results.get('methods', {}).items():
    t = entry['test']
    rows.append((name, t['selected_wer'], None))

print(f"{'method':<22} {'wer_mean':>10} {'sem':>8}")
print('-' * 44)
for name, mean, sem in rows:
    sem_s = f"{sem:>8.4f}" if sem is not None else f"{'-':>8}"
    print(f"{name:<22} {mean:>10.4f} {sem_s}")


## 6. Backup logs to Drive


In [ ]:
!cp -r reports/main_results/ami /content/drive/MyDrive/s2t-tr-dev/main_results/edinburghcstr_ami_$(date +%Y%m%d_%H%M)
